In [4]:
import os
import sys
from pathlib import Path
import json
import torch
import torch.nn as nn
from attention import self_attention, positional_encoding
import matplotlib.pyplot as plt

TOKEN_TO_IDX = {"X": 0, "O": 1, "_": 2}
RESULT_TO_IDX = {"X": 1, "O": -1, "_": 0}  # for game result
cwd = Path().resolve().parent.parent

TRAINING_DATA_FILE_PATH = "./data_generation_script/mcts_training_data.json"


In [5]:
def load_training_data(file_path):
    if not os.path.exists(file_path):
        raise ValueError("File does not exists")
    with open(file_path, "r") as f:
        data = json.load(f)

    for each in data:
        each["board_indices"] = torch.tensor(
            [TOKEN_TO_IDX[val] for val in each["board"]]
        )
        each["result_index"] = torch.tensor(RESULT_TO_IDX[each["result"]])
        each["move"] = torch.tensor(each["move"])
    return data


In [7]:
training_data = load_training_data(TRAINING_DATA_FILE_PATH)

training_data[:10]

[{'board': ['_', '_', '_', '_', '_', '_', '_', '_', '_'],
  'move': tensor(6),
  'result': 'X',
  'board_indices': tensor([2, 2, 2, 2, 2, 2, 2, 2, 2]),
  'result_index': tensor(1)},
 {'board': ['_', '_', '_', '_', '_', '_', 'X', '_', '_'],
  'move': tensor(1),
  'result': 'X',
  'board_indices': tensor([2, 2, 2, 2, 2, 2, 0, 2, 2]),
  'result_index': tensor(1)},
 {'board': ['_', 'O', '_', '_', '_', '_', 'X', '_', '_'],
  'move': tensor(0),
  'result': 'X',
  'board_indices': tensor([2, 1, 2, 2, 2, 2, 0, 2, 2]),
  'result_index': tensor(1)},
 {'board': ['X', 'O', '_', '_', '_', '_', 'X', '_', '_'],
  'move': tensor(8),
  'result': 'X',
  'board_indices': tensor([0, 1, 2, 2, 2, 2, 0, 2, 2]),
  'result_index': tensor(1)},
 {'board': ['X', 'O', '_', '_', '_', '_', 'X', '_', 'O'],
  'move': tensor(3),
  'result': 'X',
  'board_indices': tensor([0, 1, 2, 2, 2, 2, 0, 2, 1]),
  'result_index': tensor(1)},
 {'board': ['_', '_', '_', '_', '_', '_', '_', '_', '_'],
  'move': tensor(4),
  'result':

In [ ]:
policy_layer = nn.Linear(16, 1)
input = torch.randn(9, 16)
input = input.mean(dim=0)
print("input shape after mean accross column", input.shape)
policy_layer = policy_layer(input)
print("policy_Layer", policy_layer.shape)
output = torch.tanh(policy_layer)
print(input, "\n", output.shape)
output

input shape after mean accross column torch.Size([16])
policy_Layer torch.Size([1])
tensor([ 0.1288, -0.3731, -0.2164, -0.2320,  0.0177,  0.1595, -0.3763, -0.0048,
         0.2491,  0.0356, -0.1300,  0.2863,  0.0158,  0.2720,  0.0832, -0.2096]) 
 torch.Size([1])


tensor([-0.3515], grad_fn=<TanhBackward0>)

In [ ]:
policy_layer = nn.Linear(16, 1)
input = torch.randn(9, 16)
policy_layer = policy_layer(input).squeeze(1)
print("policy_Layer:", policy_layer.shape, policy_layer)
output = torch.softmax(policy_layer, dim=0)
print("output:", output.shape, output)

policy_Layer torch.Size([9]) tensor([-0.4184,  1.0865, -0.2643,  0.1175, -0.2560,  0.0276, -0.1631, -0.3704,
         0.2633], grad_fn=<SqueezeBackward1>)
output torch.Size([9]) tensor([0.0648, 0.2918, 0.0756, 0.1107, 0.0762, 0.1012, 0.0836, 0.0680, 0.1281],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
EMBEDDING_DIM = 16  # higher value for proper caputring of dim relationship
SEQ_LENGTH = 9  # board's state for a tic tac toe
VOCAB_SIZE = 3  # 3 possible board states: empty(_), X, O


class AlphaZeroTicTacToeTransformer(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.W_Q = nn.Parameter(torch.randn(EMBEDDING_DIM, EMBEDDING_DIM))
        self.W_K = nn.Parameter(torch.randn(EMBEDDING_DIM, EMBEDDING_DIM))
        self.W_V = nn.Parameter(torch.randn(EMBEDDING_DIM, EMBEDDING_DIM))
        self.embeddings = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)

        self.policy_layer = nn.Linear(EMBEDDING_DIM, 1)

        self.value_layer = nn.Linear(EMBEDDING_DIM, 1)

    def forward(self, x: torch.Tensor):
        x = self.embeddings(x)
        x = x + positional_encoding(x).to(x.dtype)
        # outputs.shape = [seq_length X embed dim]
        # attn_weights.shape = [seq_length X seq_length]
        # outputs= updates embeddings, each pos carries relation info
        # attn_weights carries strength of relationships inbetween every pair pos
        outputs, attn_weights = self_attention(x, self.W_Q, self.W_K, self.W_V)

        # value head: winning prob given board state (-1 to +1)
        value_head = outputs.mean(dim=0)
        value_head = self.value_layer(value_head)
        value_head = torch.tanh(value_head)

        # policy head: possible moves with prob, this is the outputlayer [SEQ_LEGNTH,]
        policy_head = self.policy_layer(outputs)
        policy_head = policy_head.squeeze(1)
        policy_head = torch.softmax(policy_head, dim=0)

        print("attn output shape", outputs.shape)
        print("attn weights shape", attn_weights.shape)
        return (
            policy_head,
            value_head,
            attn_weights,
        )


In [53]:
test = AlphaZeroTicTacToeTransformer()
x = torch.tensor([2, 2, 2, 2, 2, 2, 2, 2, 2])
test.forward(x)

attn output shape torch.Size([9, 16])
attn weights shape torch.Size([9, 9])


(tensor([0.1101, 0.1127, 0.1112, 0.1093, 0.1090, 0.1093, 0.1113, 0.1138, 0.1132],
        grad_fn=<SoftmaxBackward0>),
 tensor([-0.9929], grad_fn=<TanhBackward0>),
 tensor([[9.6163e-01, 3.8087e-02, 2.6140e-05, 2.1930e-07, 7.3392e-07, 5.6140e-05,
          2.0100e-04, 1.4898e-06, 8.5844e-10],
         [8.9499e-01, 1.0396e-01, 8.9215e-05, 4.9829e-07, 1.2035e-06, 1.1910e-04,
          8.3195e-04, 9.3980e-06, 3.9858e-09],
         [9.3483e-01, 6.4415e-02, 3.9020e-05, 2.0190e-07, 5.5591e-07, 6.9293e-05,
          6.3642e-04, 9.6897e-06, 5.3256e-09],
         [9.8148e-01, 1.8487e-02, 4.9221e-06, 1.9105e-08, 4.5512e-08, 4.0465e-06,
          2.4287e-05, 3.4282e-07, 3.2681e-10],
         [9.9042e-01, 9.5819e-03, 1.1933e-06, 2.2591e-09, 2.6547e-09, 1.1179e-07,
          3.3617e-07, 3.2380e-09, 3.5847e-12],
         [9.8150e-01, 1.8494e-02, 1.8842e-06, 1.6124e-09, 8.7981e-10, 2.5330e-08,
          6.3831e-08, 4.0713e-10, 2.0589e-13],
         [9.3111e-01, 6.8878e-02, 1.0123e-05, 6.7576e-09, 3.79

In [ ]:
nn.Linear(16, 1)

In [ ]:
training_data = load_training_data(TRAINING_DATA_FILE_PATH)

model = TicTacToeTransformer()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
split = int(0.8 * len(training_data))
train_data = training_data[:split]
val_data = training_data[split:]

len(training_data)

## Loss Function:
$$l = (z - v)^2 - \pi^\top \log p + c||\theta||^2$$

In [ ]:
losses = []

for epoch in range(20):
    epoch_loss = 0.0
    for data in train_data:
        x = data["board_indices"]  # add batch dimension
        y = data["move"]  # target as tensor

        logits, _ = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()  # accumulate batch loss

    # These lines are **after the inner loop**, still inside epoch loop
    avg_loss = epoch_loss / len(train_data)
    losses.append(avg_loss)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for data in val_data:
            x = data["board_indices"]
            y = data["move"]
            logits, _ = model(x)
            val_loss += criterion(logits, y).item()
        avg_val_loss = val_loss / len(val_data)
        model.train()

    print(
        f"Epoch {epoch + 1} - Loss: {avg_loss:.4f} Validation loss:{avg_val_loss:.4f}"
    )


In [ ]:
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Training Loss Curve")
plt.show()


In [ ]:
model = TicTacToeTransformer()
model.load_state_dict(torch.load("tictactoe_transformer_model.pth"))
model.eval()


def predict_move(model, board):
    indices = torch.tensor([TOKEN_TO_IDX[s] for s in board])

    model.eval()
    with torch.no_grad():
        logits, weights = model(indices)
    # mask illegal moves (non-empty cells)
    for i, cell in enumerate(board):
        if cell != "_":
            logits[i] = float("-inf")

    move = torch.argmax(logits).item()
    return move


In [ ]:
# O about to win - model should block at position 2
board = ["O", "O", "_", "X", "X", "_", "_", "_", "_"]
print(predict_move(model, board))  # should return 2 or 5

# X about to win - model should play position 2
board = ["X", "X", "_", "O", "O", "_", "_", "_", "_"]
print(predict_move(model, board))  # should return 2

In [ ]:
boards = {
    "empty": ["_", "_", "_", "_", "_", "_", "_", "_", "_"],
    "X about to win": ["X", "X", "_", "O", "O", "_", "_", "_", "_"],
    "midgame": ["X", "O", "_", "_", "X", "O", "_", "_", "_"],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, board) in zip(axes, boards.items()):
    indices = torch.tensor([TOKEN_TO_IDX[s] for s in board])
    model.eval()
    with torch.no_grad():
        logits, weights = model(indices)

    weight_matrix = weights
    cell_labels = [f"{s} ({i})" for i, s in enumerate(board)]

    im = ax.imshow(weight_matrix, cmap="hot")
    ax.set_xticks(range(9))
    ax.set_yticks(range(9))
    ax.set_xticklabels(cell_labels, fontsize=8)
    ax.set_yticklabels(cell_labels, fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
import random
from data_generation_script.uct_mcts import MCTSNode
from data_generation_script.tic_tac_toe import Tictactoe
from copy import deepcopy


mcts_agent = game.O
ai_agent = game.X

print(f"mcts_agent: {mcts_agent} and ai_agent:{ai_agent}")
results = {"X": 0, "O": 0, "draw": 0}
for _ in range(100):
    game = Tictactoe()
    while not game.is_gameover():
        if game.turn == ai_agent:
            move = predict_move(model, game.board)
            game.make_move(move)
        else:
            root = MCTSNode(state=game)
            mcts_best_move = root.best_action(simulations_number=500)
            # print("===mcts best move===", mcts_best_move)
            game.make_move(mcts_best_move)

    # print("\n\nFinal Board:")
    # game.print_board()

    if game.winner == game.X:
        results["X"] += 1
        # print("X won!")
    elif game.winner == game.O:
        results["O"] += 1
        # print("O won!")
    else:
        results["draw"] += 1
        # print("Draw!")

In [ ]:
print(f"mcts_agent: {mcts_agent} and ai_agent:{ai_agent}")
results

In [ ]:
torch.save(model.state_dict(), "tictactoe_transformer_model.pth")